# 07 — Прямой доступ к чувствительным полям

> **`vuln_class`:** `DIRECT_SENSITIVE` · **Риск:** 6/10 · **CWE-200, CWE-359** · **152-ФЗ**

Запрос вытаскивает колонки с **персональными данными** или **кредами** — пароли, СНИЛС, паспорт, номер карты, телефон, email — **без маскирования**. Сам по себе не атака, но классический канал утечки в bi-tools и отчётах аналитиков.


## 🧒 Аналогия для ребёнка

У тебя в школе есть журнал. В нём — оценки, домашние адреса
и номера телефонов учеников.

- **Плохо:** учительница ксерит **весь журнал** и раздаёт всем
  родителям — «вот, смотрите успеваемость». Заодно все узнают
  адреса и телефоны всех.
- **Хорошо:** учительница пишет каждому родителю **отдельную
  записку** только с оценками их ребёнка. Чужих данных никто
  не видит.

В БД: «весь журнал» — это `SELECT password_hash, passport, phone FROM users`.
«Маскированная записка» — `SELECT login, LEFT(phone, 3) || '***' FROM users`.


## 1. Setup — таблица клиентов с PII


In [ ]:
"""
@brief Подготовка окружения и mock-БД через in-memory SQLite.
@details
    Никаких внешних зависимостей кроме stdlib + sqlite3 (есть в Colab из коробки).
    SQLite используем как «упрощённую модель PostgreSQL» — он умеет
    почти весь стандартный SQL, что достаточно для демонстраций уязвимостей.
@note
    Реальная система работает на PostgreSQL (см. ADR-0001),
    использует pglast для AST-парсинга. Здесь, для наглядности,
    эмулируем аудитор через `re` (регулярки) и простой pattern matching.
"""
import sqlite3
import re
import time
from textwrap import dedent


def section(title):
    """@brief Печатает заголовок секции."""
    print("\n" + "=" * 72)
    print(title)
    print("=" * 72)


def show_result(rows, max_rows=10):
    """@brief Печатает результаты запроса в виде таблицы."""
    if not rows:
        print("  (нет строк)")
        return
    for i, r in enumerate(rows[:max_rows]):
        print(f"  {i + 1:>3}. {r}")
    if len(rows) > max_rows:
        print(f"  ... ещё {len(rows) - max_rows} строк")


def print_finding(f):
    """@brief Красиво печатает Finding от нашего аудитора."""
    print(f"  ⚠️  {f['rule_id']}")
    print(f"      vuln_class:  {f['vuln_class']}")
    print(f"      severity:    {f['severity']}")
    print(f"      risk_score:  {f['risk_score']}/10")
    print(f"      message:     {f['message']}")
    if f.get("evidence_refs"):
        print(f"      ссылки:      {', '.join(f['evidence_refs'])}")


def setup_clients_pii():
    conn = sqlite3.connect(":memory:")
    cur = conn.cursor()
    cur.execute("""
        CREATE TABLE clients (
            id            INTEGER PRIMARY KEY,
            login         TEXT,
            full_name     TEXT,
            passport      TEXT,          -- ⚠️ ПДн
            phone         TEXT,          -- ⚠️ ПДн
            card_number   TEXT,          -- ⚠️ платёжные данные
            password_hash TEXT           -- ⚠️ кред
        )""")
    cur.executemany(
        "INSERT INTO clients (login, full_name, passport, phone, card_number, password_hash) "
        "VALUES (?, ?, ?, ?, ?, ?)",
        [
            ("ivanov",  "Иван И.",   "4500 123456", "+79161234567", "4276 1234 5678 9012", "h_iv"),
            ("petrova", "Мария П.",  "4500 654321", "+79169876543", "5469 0001 0002 0003", "h_pet"),
            ("smith",   "John S.",   "P12345678",   "+1234567890",  "4242 4242 4242 4242", "h_sm"),
        ],
    )
    conn.commit()
    return conn


conn = setup_clients_pii()
section("Таблица clients (минус password_hash)")
show_result(conn.execute("SELECT id, login, full_name FROM clients").fetchall())


## 2. Уязвимый «отчёт» — аналитик хочет CSV для Excel


In [ ]:
##
# @brief УЯЗВИМАЯ функция: тащит ПДн без маскирования.
# @warning  Утечка СНИЛС / паспорта / карты в CSV → инцидент 152-ФЗ.
def export_clients_BAD(conn):
    sql = "SELECT login, full_name, passport, phone, card_number FROM clients"
    print(f"  SQL: {sql}")
    return conn.execute(sql).fetchall()


section("Аналитик выгружает «всех клиентов» — что попадает в CSV")
rows = export_clients_BAD(conn)
for r in rows:
    print(f"  {r}")


## 3. Аудитор Phase 1 — `R009-sensitive-columns`

Чек-лист имён колонок (мы переиспользуем словарь из ADR-0005,
раздел `kb.pii`). В проде ещё проверяется, не обёрнута ли колонка
маскирующей функцией (`coalesce`, `mask`, `digest`, `left`, `substring`).


In [ ]:
SENSITIVE_PATTERNS = [
    (r"(?i)^(password|passwd|pwd|secret|api[_-]?key|token|access[_-]?token)$",
     "critical", 8),
    (r"(?i)^(card[_-]?(number|num|no)|pan|cvv|cvc)$",
     "critical", 8),
    (r"(?i)^(passport|inn|snils|ogrn|ssn|social[_-]?security)$",
     "high", 7),
    (r"(?i)^(email|phone|mobile|tel)$",
     "medium", 5),
    (r"(?i)^(dob|birth(_?date|day))$",
     "medium", 5),
]


##
# @brief Phase 1 R009 — детект чувствительных колонок в SELECT.
# @details
#   1. Парсим SELECT-колонки (упрощённо).
#   2. Для каждой проверяем regex.
#   3. Не флагаем, если колонка обёрнута в маскирующую функцию.
def audit_R009_sensitive(sql_text):
    findings = []
    m = re.match(r"\s*SELECT\s+(.+?)\s+FROM\b", sql_text,
                 re.IGNORECASE | re.DOTALL)
    if not m:
        return findings
    columns_raw = m.group(1)
    columns = [c.strip() for c in columns_raw.split(",")]
    for col_expr in columns:
        # Если колонка завёрнута в маскирующую функцию — пропускаем
        if re.search(r"\b(coalesce|mask|digest|hash|left|substring|pgp_sym_decrypt)\s*\(",
                     col_expr, re.IGNORECASE):
            continue
        # Берём последний идентификатор как имя колонки
        last_id_match = re.findall(r"\b\w+\b", col_expr)
        if not last_id_match:
            continue
        col_name = last_id_match[-1]
        for pat, sev, score in SENSITIVE_PATTERNS:
            if re.match(pat, col_name):
                findings.append({
                    "rule_id":       "R009-sensitive-columns",
                    "vuln_class":    "DIRECT_SENSITIVE",
                    "severity":      sev, "risk_score": score,
                    "message":       f"Колонка {col_name!r} — чувствительная",
                    "evidence_refs": ["CWE-200", "CWE-359"],
                })
                break
    return findings


section("Аудитор по уязвимому SQL")
for f in audit_R009_sensitive("SELECT login, full_name, passport, phone, card_number FROM clients"):
    print_finding(f)


## 4. Безопасный отчёт — view с маскированием


In [ ]:
##
# @brief Безопасный отчёт: маскируем PII.
def export_clients_GOOD(conn):
    sql = """
        SELECT
            login,
            full_name,
            substr(passport, 1, 4) || '******'           AS passport_masked,
            substr(phone, 1, 3) || '***' || substr(phone, -2) AS phone_masked
        FROM clients
    """
    return conn.execute(sql).fetchall()


section("Маскированный CSV")
for r in export_clients_GOOD(conn):
    print(f"  {r}")


section("Аудитор по маскированному SQL")
fs = audit_R009_sensitive(export_clients_GOOD.__doc__ or """
    SELECT login, full_name,
           substr(passport, 1, 4) || '******' AS passport_masked,
           substr(phone, 1, 3) || '***' || substr(phone, -2) AS phone_masked
    FROM clients
""")
if fs:
    for f in fs:
        print_finding(f)
else:
    print("  ✅ Чувствительных колонок не найдено (всё под маскирующими функциями).")


## Итог

Мы увидели одно и то же на двух функциях:

- **Уязвимая** — украли данные / повредили БД / поднялись в правах.
- **Безопасная** — та же атака уходит в пустоту.

Между ними — **один аудитор** с конкретным правилом, которое можно
запустить детерминированно (без LLM) на каждом сгенерированном SQL.

## Куда дальше

- **Описание уязвимости (под микроскопом):** [problems/vulnerabilities/07-direct-sensitive-access/README.md](../../problems/vulnerabilities/07-direct-sensitive-access/README.md)
- **Варианты решения + почему так:** [problems/vulnerabilities/07-direct-sensitive-access/solutions.md](../../problems/vulnerabilities/07-direct-sensitive-access/solutions.md)
- **Архитектура цикла:** [docs/adr/0002-loop-architecture-langgraph.md](../../docs/adr/0002-loop-architecture-langgraph.md)
- **Гибридный аудитор (pglast + LLM):** [docs/adr/0004-hybrid-auditor-ast-plus-llm.md](../../docs/adr/0004-hybrid-auditor-ast-plus-llm.md)
